# 9) M1.4 From weather to power — Exercises

**Goals:** capacity factors from power series, hub-height scaling, a wind profile pipeline, fitting a Weibull distribution, solar position with pvlib, correct alignment of preceding-hour irradiance, and turning real weather into PyPSA `p_max_pu` profiles.

Units: m/s for wind speed, W/m² for irradiance, MW and MWh for power and energy, per unit (p.u.) for availability. Task 8 needs internet access; everything else runs offline.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import math

import numpy as np
import pandas as pd
import pvlib
from scipy import stats


def power_curve(v, v_cut_in=3.0, v_rated=12.0, v_cut_out=25.0):
    """Generic per-unit turbine output for wind speed(s) v in m/s (from the lecture)."""
    v = np.asarray(v, dtype=float)
    rising = (v ** 3 - v_cut_in ** 3) / (v_rated ** 3 - v_cut_in ** 3)
    out = np.where((v >= v_cut_in) & (v < v_rated), rising, 0.0)
    return np.where((v >= v_rated) & (v <= v_cut_out), 1.0, out)


LAT, LON = 43.51, 16.44   # Split

### Warm-ups

**1. Capacity factor of a power series**

Return the capacity factor of an hourly power series (MW) for a plant of capacity `p_nom_MW`, ignoring NaN hours.

In [2]:
def capacity_factor(power_MW: pd.Series, p_nom_MW: float) -> float:
    """Mean output divided by capacity, NaN hours ignored."""
    ### BEGIN SOLUTION
    return float(power_MW.mean() / p_nom_MW)
    ### END SOLUTION

In [3]:
s = pd.Series([0.0, 50.0, 100.0, np.nan])
assert math.isclose(capacity_factor(s, 100), 0.5), "(0 + 50 + 100) / 3 hours / 100 MW"

**2. Hub-height wind speed**

Scale wind speeds from a reference height to the hub height with the power law v(h) = v_ref (h / h_ref)^α. It must work on arrays.

In [4]:
def hub_height_speed(v_ref, h_ref: float, h_hub: float, alpha: float = 1 / 7):
    """Wind speed (m/s) at hub height h_hub from speeds measured at h_ref."""
    ### BEGIN SOLUTION
    return np.asarray(v_ref, dtype=float) * (h_hub / h_ref) ** alpha
    ### END SOLUTION

In [5]:
assert np.allclose(hub_height_speed([5.0, 10.0], 100, 100), [5.0, 10.0]), "same height -> same speed"
assert math.isclose(float(hub_height_speed(5.0, 10, 100)), 5 * 10 ** (1 / 7))
assert float(hub_height_speed(5.0, 100, 150, alpha=0.2)) > 5.0, "higher hub -> faster wind"

### Core

**3. Wind availability profile**

Combine tasks 2 and the lecture's `power_curve`: from wind speeds at 100 m, return per-unit availability for a turbine with hub height `h_hub` (index preserved). This is exactly what PyPSA needs as `p_max_pu`.

In [6]:
def wind_pu(v_100m: pd.Series, h_hub: float = 120, alpha: float = 1 / 7) -> pd.Series:
    """Per-unit wind availability from wind speeds at 100 m."""
    ### BEGIN SOLUTION
    v_hub = hub_height_speed(v_100m.to_numpy(), 100, h_hub, alpha)
    return pd.Series(power_curve(v_hub), index=v_100m.index, name="wind_pu")
    ### END SOLUTION

In [7]:
v = pd.Series([2.0, 6.0, 12.0, 30.0], index=pd.date_range("2025-01-01", periods=4, freq="h", tz="UTC"))
w = wind_pu(v, h_hub=100)
assert w.index.equals(v.index), "keep the timestamps"
assert np.allclose(w, power_curve(v)), "at 100 m hub height no scaling"
assert w.iloc[3] == 0, "above cut-out the turbine stops"
assert (wind_pu(v, h_hub=150) >= w - 1e-12).all(), "a taller hub never produces less"

**4. Fit a Weibull distribution**

Given hourly wind speeds, fit a Weibull distribution (location fixed at 0) and return shape k and scale c. Check it on a synthetic sample with known parameters, then compare the capacity factor from the *fitted* distribution with the one from the hourly data directly.

In [8]:
def fit_weibull(speeds) -> tuple[float, float]:
    """Return (k, c) of a two-parameter Weibull fit to wind speeds (m/s)."""
    ### BEGIN SOLUTION
    k, _, c = stats.weibull_min.fit(np.asarray(speeds, dtype=float), floc=0)
    return float(k), float(c)
    ### END SOLUTION

In [9]:
sample = stats.weibull_min.rvs(2.2, scale=7.5, size=20_000, random_state=np.random.default_rng(1))
k, c = fit_weibull(sample)
assert abs(k - 2.2) / 2.2 < 0.05 and abs(c - 7.5) / 7.5 < 0.05, "recover the true parameters within 5 %"
cf_direct = power_curve(sample).mean()
cf_fitted = power_curve(stats.weibull_min.rvs(k, scale=c, size=200_000, random_state=np.random.default_rng(2))).mean()
assert abs(cf_direct - cf_fitted) < 0.01
print(f"k = {k:.2f}, c = {c:.2f} m/s, CF direct {cf_direct:.3f}, CF from fit {cf_fitted:.3f}")

k = 2.20, c = 7.49 m/s, CF direct 0.254, CF from fit 0.253


**5. Preceding-hour irradiance and the sun's position**

Open-Meteo's GHI stamped at time t is the mean over (t − 1 h, t]. To compute the sun's position for that value you need the **middle** of the interval, t − 30 min. Write `interval_midpoints` and `clear_sky_pu`: per-unit clear-sky PV output (horizontal, performance ratio `pr`) for end-of-interval timestamps, using pvlib's solar position and Haurwitz clear-sky model.

In [10]:
def interval_midpoints(end_stamps: pd.DatetimeIndex) -> pd.DatetimeIndex:
    """Midpoints of hourly intervals labelled by their end time."""
    ### BEGIN SOLUTION
    return end_stamps - pd.Timedelta("30min")
    ### END SOLUTION


def clear_sky_pu(end_stamps: pd.DatetimeIndex, lat: float, lon: float, pr: float = 0.86) -> pd.Series:
    """Per-unit clear-sky PV output for hourly intervals labelled by their end time."""
    ### BEGIN SOLUTION
    mid = interval_midpoints(end_stamps)
    zenith = pvlib.solarposition.get_solarposition(mid, lat, lon)["apparent_zenith"]
    ghi = pvlib.clearsky.haurwitz(zenith)["ghi"].fillna(0.0).to_numpy()
    return pd.Series(np.clip(ghi / 1000 * pr, 0, 1), index=end_stamps, name="solar_pu")
    ### END SOLUTION

In [11]:
ends = pd.date_range("2025-01-01 01:00", periods=8760, freq="h", tz="UTC")
assert interval_midpoints(ends)[0] == pd.Timestamp("2025-01-01 00:30", tz="UTC")
cs = clear_sky_pu(ends, LAT, LON)
assert cs.index.equals(ends)
assert cs.between(0, 1).all() and cs[ends.hour == 1].max() == 0, "night hours are zero"
assert 0.17 < cs.mean() < 0.22, "clear-sky CF near Split, horizontal: about 0.19-0.20"
print(f"clear-sky capacity factor: {cs.mean():.3f}")

clear-sky capacity factor: 0.195


**6. Interpretation**

Your clear-sky capacity factor is an upper bound for horizontal PV at this site. Give two reasons real PV capacity factors are lower, and one reason a real installation could reach a *higher* capacity factor than your horizontal model. Then: solar energy arrives mostly around midday in summer. Why might a MWh of solar be worth less on the market than a MWh of wind? (A preview of Module 02.)

<!-- BEGIN SOLUTION -->
Lower in reality: **clouds and haze** reduce irradiance on most days; **high cell temperatures** in summer reduce efficiency more than the flat performance ratio assumes; plus snow, soiling, shading, outages and inverter clipping. Higher than the model: real panels are **tilted** (and some **track** the sun), so they face the sun more directly than a horizontal surface — fixed tilt near the latitude raises annual yield noticeably, trackers more. On value: all solar plants in a region produce at the same time, so when solar capacity is large, midday supply rises and the merit-order effect (M1.2) pushes midday prices down — sometimes below zero. A MWh of solar is then sold mostly in cheap hours (**price cannibalisation**), while wind produces at more varied times, including evenings and winter. Capacity factor measures quantity; market value also depends on *when* the energy arrives.
<!-- END SOLUTION -->

### Challenge

**7. From weather to PyPSA profiles**

Write `profiles_from_weather`: from a DataFrame with columns `wind_100m_m_per_s` and `ghi_W_per_m2` (index = interval **end** times, UTC), return a DataFrame with `wind_pu` and `solar_pu`, both in [0, 1], **indexed by interval start** (end − 1 h) — the convention PyPSA snapshots and ENTSO-E hourly data use. Test it offline on a small weather table with values you can check by hand.

In [12]:
def profiles_from_weather(weather: pd.DataFrame, h_hub: float = 120, alpha: float = 1 / 7,
                          pr: float = 0.86) -> pd.DataFrame:
    """Per-unit wind and solar availability, indexed by interval start (UTC)."""
    ### BEGIN SOLUTION
    w = wind_pu(weather["wind_100m_m_per_s"], h_hub, alpha)
    s = np.clip(weather["ghi_W_per_m2"].to_numpy() / 1000 * pr, 0, 1)
    out = pd.DataFrame({"wind_pu": w.to_numpy(), "solar_pu": s}, index=weather.index - pd.Timedelta("1h"))
    return out
    ### END SOLUTION

In [13]:
test_weather = pd.DataFrame({"wind_100m_m_per_s": [2.0, 8.0, 12.0, 30.0],
                             "ghi_W_per_m2": [0.0, 500.0, 1000.0, 1300.0]},
                            index=pd.date_range("2025-06-01 10:00", periods=4, freq="h", tz="UTC"))
prof = profiles_from_weather(test_weather, h_hub=100, pr=0.8)
assert prof.index[0] == pd.Timestamp("2025-06-01 09:00", tz="UTC"), "label by interval start"
assert np.allclose(prof["solar_pu"], [0.0, 0.4, 0.8, 1.0]), "GHI/1000 x PR, capped at 1"
assert np.allclose(prof["wind_pu"], power_curve([2.0, 8.0, 12.0, 30.0]))
assert prof.to_numpy().min() >= 0 and prof.to_numpy().max() <= 1

**8. Real 2024 profiles for Split**

Download 2024 hourly weather for Split with Open-Meteo (internet needed), build the profiles, and report the wind and solar capacity factors. The solar value must stay below your clear-sky bound from task 5 — if it doesn't, look for a unit or alignment error before anything else.

In [14]:
def fetch_open_meteo(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Hourly ERA5 wind speed at 100 m (m/s) and GHI (W/m2, preceding-hour mean) from Open-Meteo."""
    import requests
    params = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end,
              "hourly": "wind_speed_100m,shortwave_radiation", "wind_speed_unit": "ms", "timezone": "UTC"}
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
    r.raise_for_status()
    h = r.json()["hourly"]
    return pd.DataFrame({"wind_100m_m_per_s": h["wind_speed_100m"], "ghi_W_per_m2": h["shortwave_radiation"]},
                        index=pd.to_datetime(h["time"], utc=True))


try:
    split_weather = fetch_open_meteo(LAT, LON, "2024-01-01", "2024-12-31")
except Exception as exc:
    split_weather = None
    print(f"Skipped (no data downloaded): {type(exc).__name__}")

if split_weather is not None:
    split_prof = profiles_from_weather(split_weather)
    bound = clear_sky_pu(split_weather.index, LAT, LON).mean()
    print(f"wind CF {split_prof.wind_pu.mean():.3f}, solar CF {split_prof.solar_pu.mean():.3f} "
          f"(clear-sky bound {bound:.3f})")
    assert split_prof.solar_pu.mean() < bound, "real solar CF above the clear-sky bound: check units/alignment"

Skipped (no data downloaded): ConnectionError


**9. How much can you trust these profiles?**

ERA5 is a reanalysis on a grid of roughly 30 km. Name two ways its wind speeds can differ from what a real turbine on a specific Croatian ridge or coastline experiences, and what a developer would do about it before investing.

<!-- BEGIN SOLUTION -->
A ~30 km grid cell averages over very different terrain, so it **smooths local effects**: speed-up over ridges and hills, channelling through valleys, the strong gusty downslope winds typical of parts of the Adriatic coast, and the transition between sea and land are all poorly resolved. Wind speeds at a specific site can therefore be systematically higher or lower than ERA5 (a **bias**), and the shape of the distribution — especially the high-speed tail that matters most under the cube law — can be wrong. Reanalysis also models, rather than measures, conditions at 100 m. Before investing, developers install a **measurement mast or lidar** on site for at least a year, compare it with the reanalysis over the same period, and use that relationship to **bias-correct** long-term reanalysis data (measure–correlate–predict), so they get both local accuracy and decades of climate variability. For solar, ground measurements or satellite-based datasets with higher resolution play the same role.
<!-- END SOLUTION -->